# Part 1 – Exploratory data analysis

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("porto/porto/porto.csv")

n_points = df["POLYLINE"].str.count(r"\[") - 1
duration_min = (n_points - 1).clip(lower=0) * 15 / 60
start_time = pd.to_datetime(df["TIMESTAMP"], unit="s", utc=True).dt.tz_convert("Europe/Lisbon")


def show_counts(s, order, decimals=2):
    for key, n in s.value_counts().reindex(order, fill_value=0).items():
        print(f"{key}: {n:>9,} ({n / len(s):.{decimals}%})")


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6_371_000 * np.arcsin(np.sqrt(a))

## Overview

In [2]:
print(f"{len(df):,} rows, {df.shape[1]} columns")
width = max(map(len, df.columns))
for col, dtype in df.dtypes.items():
    print(f"  {col:<{width}}  {dtype}")
print(f"First trip: {start_time.min():%Y-%m-%d %H:%M} (Porto time)")
print(f"Last trip:  {start_time.max():%Y-%m-%d %H:%M} (Porto time)")

1,710,670 rows, 9 columns
  TRIP_ID       int64
  CALL_TYPE     str
  ORIGIN_CALL   float64
  ORIGIN_STAND  float64
  TAXI_ID       int64
  TIMESTAMP     int64
  DAY_TYPE      str
  MISSING_DATA  bool
  POLYLINE      str
First trip: 2013-07-01 01:00 (Porto time)
Last trip:  2014-07-01 00:59 (Porto time)


## Duplicates

In [3]:
dupes = df[df["TRIP_ID"].duplicated(keep=False)].sort_values("TRIP_ID")

print(f"Duplicate TRIP_IDs:    {df['TRIP_ID'].duplicated().sum()} ({len(dupes)} rows involved)")
print(f"Exact duplicate rows:  {df.duplicated().sum()}")

exact = df[df.duplicated(keep=False)].sort_values("TRIP_ID")
same_id = dupes[~dupes.duplicated(keep=False)]

with pd.option_context("display.max_colwidth", 100):
    print("Exact duplicate rows")
    display(exact.assign(POINTS=n_points))
    print("Same TRIP_ID, different content (first 5 pairs)")
    display(same_id.assign(POINTS=n_points).head(10))

Duplicate TRIP_IDs:    81 (161 rows involved)
Exact duplicate rows:  3
Exact duplicate rows


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE,POINTS
302435,1378607552620000403,C,NaN,NaN,20000403,1378607552,A,False,[],0
302441,1378607552620000403,C,NaN,NaN,20000403,1378607552,A,False,[],0
998727,1391416612620000196,C,NaN,NaN,20000196,1391416612,A,False,"[[-8.555265,41.180472]]",1
998728,1391416612620000196,C,NaN,NaN,20000196,1391416612,A,False,"[[-8.555265,41.180472]]",1
1304114,1397172149620000454,C,NaN,NaN,20000454,1397172149,A,False,"[[-8.618391,41.162067],[-8.6184,41.161446],[-8.619111,41.162292],[-8.618391,41.162454],[-8.61837...",6
1304115,1397172149620000454,C,NaN,NaN,20000454,1397172149,A,False,"[[-8.618391,41.162067],[-8.6184,41.161446],[-8.619111,41.162292],[-8.618391,41.162454],[-8.61837...",6


Same TRIP_ID, different content (first 5 pairs)


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE,POINTS
3519,1372702836620000080,B,NaN,16.0,20000080,1372702836,A,False,"[[-8.635275,41.193729],[-8.634519,41.192901]]",2
3720,1372702836620000080,C,NaN,NaN,20000080,1372702836,A,False,"[[-8.62776,41.17554],[-8.62776,41.17554],[-8.627778,41.175531],[-8.627796,41.175504],[-8.627796,...",54
21282,1373025987620000601,A,2002.0,NaN,20000601,1373025987,A,False,[],0
21385,1373025987620000601,C,NaN,NaN,20000601,1373025987,A,False,"[[-8.604108,41.182569],[-8.604072,41.182605],[-8.604036,41.182614],[-8.604045,41.182569],[-8.604...",37
33228,1373210896620000598,B,NaN,57.0,20000598,1373210896,A,False,"[[-8.610885,41.145525]]",1
33350,1373210896620000598,C,NaN,NaN,20000598,1373210896,A,False,"[[-8.610813,41.14548],[-8.610732,41.145579],[-8.610327,41.146146],[-8.610255,41.146236],[-8.6094...",19
51623,1373551710620000014,B,NaN,49.0,20000014,1373551710,A,False,"[[-8.587278,41.147325],[-8.587278,41.147325]]",2
51966,1373551710620000014,C,NaN,NaN,20000014,1373551710,A,False,"[[-8.602938,41.179716],[-8.603937,41.179932],[-8.604747,41.180193],[-8.604738,41.180184],[-8.605...",67
76527,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648451,41.164785]]",1
76596,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648478,41.164758],[-8.648478,41.164758],[-8.64846,41.164758],[-8.649099,41.164677],[-8.6502...",75


## Call type

In [4]:
show_counts(df["CALL_TYPE"], ["A", "B", "C"])

has_call = df["ORIGIN_CALL"].notna()
has_stand = df["ORIGIN_STAND"].notna()
valid = (
    ((df["CALL_TYPE"] == "A") & has_call & ~has_stand)
    | ((df["CALL_TYPE"] == "B") & ~has_call & has_stand)
    | ((df["CALL_TYPE"] == "C") & ~has_call & ~has_stand)
)
mismatch = df.loc[~valid, "CALL_TYPE"].value_counts().reindex(["A", "B", "C"], fill_value=0)

print(f"\nMismatch between call type and origin: {(~valid).sum():,}")
for call_type, n in mismatch.items():
    print(f"{call_type}: {n:>9,}")
df[~valid].drop(columns="POLYLINE").head()

A:   364,770 (21.32%)
B:   817,881 (47.81%)
C:   528,019 (30.87%)

Mismatch between call type and origin: 11,302
A:         0
B:    11,302
C:         0


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA
136,1372638560620000308,B,NaN,NaN,20000308,1372638560,A,False
280,1372650696620000320,B,NaN,NaN,20000320,1372650696,A,False
433,1372656129620000307,B,NaN,NaN,20000307,1372656129,A,False
564,1372665032620000004,B,NaN,NaN,20000004,1372665032,A,False
640,1372663170620000391,B,NaN,NaN,20000391,1372663170,A,False


## Day type

In [5]:
show_counts(df["DAY_TYPE"], ["A", "B", "C"])

A: 1,710,670 (100.00%)
B:         0 (0.00%)
C:         0 (0.00%)


## Missing data flag

In [6]:
show_counts(df["MISSING_DATA"], [False, True], decimals=4)

rows = []
for trip_id, polyline in df.loc[df["MISSING_DATA"], ["TRIP_ID", "POLYLINE"]].itertuples(index=False):
    t = np.array(json.loads(polyline))
    seg = pd.Series(haversine(t[:-1, 1], t[:-1, 0], t[1:, 1], t[1:, 0]))
    rows.append({
        "TRIP_ID": trip_id,
        "Points": len(t),
        "Max jump (m)": seg.max(),
        "Median jump (m)": seg.median(),
        "Jumps > 1 km": (seg > 1000).sum() if len(seg) else np.nan,
    })

print("\nDistance between consecutive points in flagged trips")
pd.DataFrame(rows).sort_values("Max jump (m)", ascending=False).round(0).astype("Int64")

False: 1,710,660 (99.9994%)
True:        10 (0.0006%)

Distance between consecutive points in flagged trips


,TRIP_ID,Points,Max jump (m),Median jump (m),Jumps > 1 km
3,1381233613620000387,483,43870,10,1
4,1386346894620000904,285,11003,553,1
2,1378544246620000057,11,8399,240,2
9,1399405185620000508,53,5174,130,1
5,1387137779620000640,51,2245,140,1
7,1390005983620000640,14,789,31,0
6,1388351478620000678,27,778,114,0
0,1374554455620000625,43,354,103,0
1,1375863510620000454,1,<NA>,<NA>,<NA>
8,1396631707620000163,1,<NA>,<NA>,<NA>


## Taxis

In [7]:
per_taxi = df["TAXI_ID"].value_counts()

print(f"Taxis: {len(per_taxi)}")
print(f"Trips per taxi: mean {per_taxi.mean():.0f}, median {per_taxi.median():.0f}, "
      f"min {per_taxi.min()}, max {per_taxi.max()}")

Taxis: 448
Trips per taxi: mean 3818, median 3732, min 1, max 10746


## Time patterns (Porto local time)

In [8]:
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

print("Trips per hour")
print(start_time.dt.hour.value_counts().sort_index().rename_axis(None).to_string())
print("\nTrips per weekday")
print(start_time.dt.day_name().value_counts().reindex(weekdays).rename_axis(None).to_string())
print("\nTrips per month")
print(start_time.dt.strftime("%Y-%m").value_counts().sort_index().rename_axis(None).to_string())

Trips per hour
0     49626
1     49881
2     49279
3     47677
4     61007
5     56450
6     52608
7     54291
8     82372
9     99039
10    95389
11    88675
12    84240
13    82397
14    93970
15    90349
16    92010
17    87246
18    80842
19    74025
20    69903
21    63289
22    56008
23    50097

Trips per weekday
Monday       229345
Tuesday      235111
Wednesday    232010
Thursday     246641
Friday       275113
Saturday     258064
Sunday       234386

Trips per month
2013-07    145501
2013-08    125804
2013-09    147600
2013-10    153504
2013-11    140024
2013-12    146258
2014-01    130875
2014-02    129872
2014-03    138905
2014-04    136931
2014-05    161982
2014-06    153288
2014-07       126


## Trajectories: points and duration

In [9]:
stats = pd.DataFrame({"points": n_points, "minutes": duration_min})
print(stats.agg(["mean", "std", "min", "max"]).round(1).to_string())

print(f"\nEmpty:               {(n_points == 0).sum():,}")
print(f"Fewer than 3 points: {(n_points < 3).sum():,}")
print(f"Over 2 hours:        {(duration_min > 120).sum():,}")
print(f"Over 5 hours:        {(duration_min > 300).sum():,}")

      points  minutes
mean    48.8     11.9
std     45.7     11.4
min      0.0      0.0
max   3881.0    970.0

Empty:               5,901
Fewer than 3 points: 43,904
Over 2 hours:        2,257
Over 5 hours:        208


## Trajectories: coordinates and distance (sample of 50,000 trips)

In [17]:
sample = df.loc[n_points > 0, "POLYLINE"].sample(50_000, random_state=1)
trajs = [np.array(json.loads(p)) for p in sample]
lengths = np.array([len(t) for t in trajs])

lon, lat = np.concatenate(trajs).T
trip = np.repeat(np.arange(len(trajs)), lengths)

same = trip[1:] == trip[:-1]
seg = haversine(lat[:-1], lon[:-1], lat[1:], lon[1:])[same]
seg_trip = trip[:-1][same]
jump = seg > 625  # faster than 150 km/h over 15 s
print(f"GPS jumps over 625 m: {jump.sum():,} in {len(np.unique(seg_trip[jump])):,} trips")

km = pd.Series(np.bincount(seg_trip, weights=seg, minlength=len(trajs)) / 1000)
print("\nDistance per trip (km)")
print(km.agg(["mean", "std", "min", "max"]).round(2).to_string())

GPS jumps over 625 m: 6,771 in 5,088 trips

Distance per trip (km)
mean      5.51
std       7.33
min       0.00
max     574.86


## Impact of missing trajectories

In [18]:
empty = n_points == 0
short = n_points < 3
flagged = df["MISSING_DATA"]
too_long = duration_min > 120

print(f"Empty polylines flagged MISSING_DATA: {(empty & flagged).sum()} of {empty.sum():,}")

print("\nShare of each call type (%)")
share = pd.DataFrame({"empty": empty, "< 3 points": short, "> 2 hours": too_long})
print(share.groupby(df["CALL_TYPE"]).mean().mul(100).round(2).rename_axis(None).to_string())

print("\nLost if removed")
rules = {
    "< 3 points": short,
    "MISSING_DATA": flagged,
    "> 2 hours": too_long,
    "All combined": short | flagged | too_long,
}
for name, mask in rules.items():
    pts = n_points[mask].sum()
    print(f"{name:<13} {mask.sum():>7,} trips ({mask.mean():.2%})  "
          f"{pts:>10,} points ({pts / n_points.sum():.2%})")

Empty polylines flagged MISSING_DATA: 0 of 5,901

Share of each call type (%)
   empty  < 3 points  > 2 hours
A   0.12        0.71       0.04
B   0.22        1.28       0.04
C   0.69        5.85       0.34

Lost if removed
< 3 points     43,904 trips (2.57%)      45,397 points (0.05%)
MISSING_DATA       10 trips (0.00%)         969 points (0.00%)
> 2 hours       2,257 trips (0.13%)   1,740,503 points (2.09%)
All combined   46,168 trips (2.70%)   1,786,384 points (2.14%)


## Plots

In [ ]:
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlelocation": "left",
    "axes.axisbelow": True,
    "grid.color": "#e6e5e1",
})
blue = "#2a78d6"
fig, ax = plt.subplots(2, 2, figsize=(13, 10))

calls = df["CALL_TYPE"].value_counts().sort_index()
bars = ax[0, 0].bar(calls.index, calls.values, color=blue, width=0.6)
ax[0, 0].bar_label(bars, labels=[f"{n / len(df):.1%}" for n in calls], padding=3)
ax[0, 0].set(title="Trips per call type", ylabel="Trips")
ax[0, 0].grid(axis="y")

hours = start_time.dt.hour.value_counts().sort_index()
ax[0, 1].bar(hours.index, hours.values, color=blue)
ax[0, 1].set(title="Trips per hour (Porto time)", xlabel="Hour", ylabel="Trips", xticks=range(0, 24, 2))
ax[0, 1].grid(axis="y")

ax[1, 0].hist(duration_min[duration_min <= 60], bins=60, color=blue, edgecolor="white", linewidth=0.5)
ax[1, 0].set(title=f"Trip duration up to 60 min ({(duration_min > 60).sum():,} longer not shown)",
             xlabel="Minutes", ylabel="Trips")
ax[1, 0].grid(axis="y")

starts = np.array([t[0] for t in trajs])
ax[1, 1].scatter(starts[:, 0], starts[:, 1], s=1, alpha=0.2, color=blue)
ax[1, 1].set(title="Start points (sample)", xlabel="Longitude", ylabel="Latitude",
             xlim=(-8.75, -8.45), ylim=(41.05, 41.30))
ax[1, 1].set_aspect(1 / np.cos(np.radians(41.15)))

plt.tight_layout()
plt.show()